In [3]:
from google.colab import files
import pandas as pd

# Abre el explorador de archivos local
uploaded = files.upload()

Saving DATASET_TAREA_2_CONSOLIDADO_FINAL_ORIGINAL.xlsx to DATASET_TAREA_2_CONSOLIDADO_FINAL_ORIGINAL.xlsx


In [6]:
# -*- coding: utf-8 -*-
"""
Comparación LightGBM + Red Neuronal Básica vs DeepSeek API con prompting.

CORRECCIONES:
- NO usa train_test_split
- Usa columna Dataset_type:
    * TRAIN
    * TEST
    * EVAL
- Entrena SOLO con TRAIN
- Evalúa sobre TEST y EVAL
- DeepSeek usa TODA la población EVAL
- Mantiene las 3 estrategias prompting:
    * zero-shot
    * few-shot
    * cot
- Usa 30 ejemplos de prompting
"""

# =========================================================
# 0. INSTALAR LIBRERIAS (SOLO COLAB)
# =========================================================

# !pip install lightgbm openpyxl -q

# =========================================================
# 1. LIBRERIAS
# =========================================================

import pandas as pd
import numpy as np
import requests
import scipy.sparse as sp
import time

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MaxAbsScaler

import lightgbm as lgb

# =========================================================
# 2. CARGAR DATASET
# =========================================================

data = pd.read_excel(
    "DATASET_TAREA_2_CONSOLIDADO_FINAL_ORIGINAL.xlsx"
)

# =========================================================
# 3. LIMPIEZA BASICA
# =========================================================

data = data.dropna(
    subset=[
        "text",
        "label",
        "pos_rel",
        "Dataset_type"
    ]
)

data["text"] = data["text"].astype(str)

data["Dataset_type"] = (
    data["Dataset_type"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# =========================================================
# 4. LABEL BINARIO
# =========================================================

data["label"] = np.where(
    data["label"] == "CONTR",
    1,
    0
)

data["label"] = data["label"].astype(int)

print("\n===================================")
print("Distribucion de clases")
print("===================================")

print(
    data["label"].value_counts()
)

print("\n===================================")
print("Distribucion Dataset_type")
print("===================================")

print(
    data["Dataset_type"].value_counts()
)

# =========================================================
# 5. DIVISIONES DEL DATASET
# =========================================================

train_data = data[
    data["Dataset_type"] == "TRAIN"
]

test_data = data[
    data["Dataset_type"] == "TEST"
]

eval_data = data[
    data["Dataset_type"] == "EVAL"
]

print("\n===================================")
print("TRAIN")
print("===================================")
print(len(train_data))

print("\n===================================")
print("TEST")
print("===================================")
print(len(test_data))

print("\n===================================")
print("EVAL")
print("===================================")
print(len(eval_data))

# =========================================================
# 6. TF-IDF
# =========================================================

vectorizer = TfidfVectorizer(
    max_features=5000
)

X_train_text = vectorizer.fit_transform(
    train_data["text"]
)

X_test_text = vectorizer.transform(
    test_data["text"]
)

X_eval_text = vectorizer.transform(
    eval_data["text"]
)

# =========================================================
# 7. FEATURE POS_REL
# =========================================================

X_train_pos = (
    train_data["pos_rel"]
    .values
    .reshape(-1, 1)
)

X_test_pos = (
    test_data["pos_rel"]
    .values
    .reshape(-1, 1)
)

X_eval_pos = (
    eval_data["pos_rel"]
    .values
    .reshape(-1, 1)
)

# =========================================================
# 8. MATRICES FINALES
# =========================================================

X_train = sp.hstack([
    X_train_text,
    X_train_pos
])

X_test = sp.hstack([
    X_test_text,
    X_test_pos
])

X_eval = sp.hstack([
    X_eval_text,
    X_eval_pos
])

y_train = train_data["label"]

y_test = test_data["label"]

y_eval = eval_data["label"]

# =========================================================
# 8.1 ESCALADO PARA RED NEURONAL
# =========================================================

scaler = MaxAbsScaler()

X_train_nn = scaler.fit_transform(X_train)

X_test_nn = scaler.transform(X_test)

X_eval_nn = scaler.transform(X_eval)

# =========================================================
# 9. FUNCION METRICAS
# =========================================================

def evaluar_modelo(
    nombre,
    y_true,
    y_pred,
    y_proba=None
):

    print("\n================================================")
    print(nombre)
    print("================================================")

    print(
        "Accuracy :",
        round(
            accuracy_score(y_true, y_pred),
            4
        )
    )

    print(
        "Precision:",
        round(
            precision_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),
            4
        )
    )

    print(
        "Recall   :",
        round(
            recall_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),
            4
        )
    )

    print(
        "F1 Score :",
        round(
            f1_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),
            4
        )
    )

    if y_proba is not None:

        try:

            roc = roc_auc_score(
                y_true,
                y_proba
            )

            print(
                "ROC-AUC  :",
                round(roc, 4)
            )

        except Exception as e:

            print("ROC-AUC no disponible")
            print(e)

    print("\nReporte completo:\n")

    print(
        classification_report(
            y_true,
            y_pred,
            zero_division=0
        )
    )

# =========================================================
# 10. LIGHTGBM
# =========================================================

print("\nEntrenando LightGBM...")

lgb_model = lgb.LGBMClassifier(
    random_state=42
)

lgb_model.fit(
    X_train,
    y_train
)

# =========================================================
# 11. EVALUACION TEST
# =========================================================

print("\n########################################")
print("EVALUACION TEST")
print("########################################")

y_pred_test = lgb_model.predict(
    X_test
)

y_proba_test = lgb_model.predict_proba(
    X_test
)[:, 1]

evaluar_modelo(
    "LightGBM TEST",
    y_test,
    y_pred_test,
    y_proba_test
)

# =========================================================
# 12. EVALUACION EVAL
# =========================================================

print("\n########################################")
print("EVALUACION EVAL")
print("########################################")

y_pred_eval = lgb_model.predict(
    X_eval
)

y_proba_eval = lgb_model.predict_proba(
    X_eval
)[:, 1]

evaluar_modelo(
    "LightGBM EVAL",
    y_eval,
    y_pred_eval,
    y_proba_eval
)

# =========================================================
# 12.1 RED NEURONAL BASICA
# =========================================================

print("\nEntrenando Red Neuronal Basica...")

nn_model = MLPClassifier(

    hidden_layer_sizes=(),

    activation="logistic",

    solver="adam",

    max_iter=300,

    random_state=42
)

nn_model.fit(
    X_train_nn,
    y_train
)

# =========================================================
# RED NEURONAL TEST
# =========================================================

print("\n########################################")
print("RED NEURONAL BASICA TEST")
print("########################################")

y_pred_nn_test = nn_model.predict(
    X_test_nn
)

y_proba_nn_test = nn_model.predict_proba(
    X_test_nn
)[:, 1]

evaluar_modelo(
    "Red Neuronal Basica TEST",
    y_test,
    y_pred_nn_test,
    y_proba_nn_test
)

# =========================================================
# RED NEURONAL EVAL
# =========================================================

print("\n########################################")
print("RED NEURONAL BASICA EVAL")
print("########################################")

y_pred_nn_eval = nn_model.predict(
    X_eval_nn
)

y_proba_nn_eval = nn_model.predict_proba(
    X_eval_nn
)[:, 1]

evaluar_modelo(
    "Red Neuronal Basica EVAL",
    y_eval,
    y_pred_nn_eval,
    y_proba_nn_eval
)

# =========================================================
# 13. 30 EJEMPLOS PROMPTING
# =========================================================

PROMPT_EXAMPLES = {

    "zero-shot": {

        "positivos": [
            {
                "texto": "los resultados obtenidos contradicen la hipotesis inicial debido a que no se encontro relacion significativa entre las variables de estudio en ninguno de los grupos evaluados",
                "label": "CONTR"
            },
            {
                "texto": "a diferencia de investigaciones previas el modelo aplicado no mejoro el rendimiento academico y presento resultados inferiores en comparacion con el metodo tradicional",
                "label": "CONTR"
            },
            {
                "texto": "los hallazgos muestran que la implementacion de estrategias agroecologicas no produjo el aumento esperado en la productividad de los cultivos evaluados",
                "label": "CONTR"
            },
            {
                "texto": "el analisis estadistico evidencio que la intervencion educativa aplicada no genero cambios significativos en las competencias ciudadanas de los estudiantes",
                "label": "CONTR"
            },
            {
                "texto": "los datos recopilados durante la investigacion refutan la teoria planteada inicialmente acerca de la relacion entre mineria y desarrollo institucional",
                "label": "CONTR"
            }
        ],

        "negativos": [
            {
                "texto": "los resultados permitieron confirmar que el uso de tecnologias educativas favorece el aprendizaje colaborativo en ambientes virtuales",
                "label": "NO CONTR"
            },
            {
                "texto": "el estudio concluye que la implementacion de sistemas de control interno mejora la eficiencia administrativa y reduce riesgos financieros",
                "label": "NO CONTR"
            },
            {
                "texto": "la investigacion demuestra que la diversidad vegetal en sistemas agroecologicos favorece la presencia de organismos beneficos",
                "label": "NO CONTR"
            },
            {
                "texto": "los autores encontraron evidencia suficiente para respaldar la relacion positiva entre participacion familiar y desarrollo cognitivo infantil",
                "label": "NO CONTR"
            },
            {
                "texto": "el analisis de los datos confirma que las estrategias pedagogicas aplicadas fortalecieron el proceso de aprendizaje de los estudiantes",
                "label": "NO CONTR"
            }
        ]
    },

    "few-shot": {

        "positivos": [
            {
                "texto": "contrario a lo reportado en investigaciones anteriores la aplicacion del modelo matematico no permitio mejorar la estabilidad del vehiculo",
                "label": "CONTR"
            },
            {
                "texto": "los resultados experimentales no respaldan la teoria de que el incremento de concesiones mineras fortalece la institucionalidad regional",
                "label": "CONTR"
            },
            {
                "texto": "el estudio evidencio que las practicas de control propuestas no redujeron los costos operativos como se esperaba inicialmente",
                "label": "CONTR"
            },
            {
                "texto": "a pesar de las expectativas iniciales el uso de plataformas virtuales no incremento la participacion activa de los estudiantes en los foros",
                "label": "CONTR"
            },
            {
                "texto": "la investigacion concluye que la intervencion aplicada no tuvo impacto significativo sobre la reduccion de accidentes de motociclistas",
                "label": "CONTR"
            }
        ],

        "negativos": [
            {
                "texto": "los hallazgos respaldan la importancia de incorporar estrategias de responsabilidad social empresarial en las politicas gubernamentales",
                "label": "NO CONTR"
            },
            {
                "texto": "los resultados obtenidos muestran una mejora considerable en la interaccion social de los estudiantes despues de aplicar la estrategia educativa",
                "label": "NO CONTR"
            },
            {
                "texto": "el estudio confirma que la utilizacion de sensores inteligentes mejora la estabilidad y seguridad en los sistemas de control vehicular",
                "label": "NO CONTR"
            },
            {
                "texto": "la investigacion evidencia que los sistemas agroecologicos presentan una mayor diversidad biologica que los sistemas convencionales",
                "label": "NO CONTR"
            },
            {
                "texto": "los autores concluyen que el fortalecimiento de las competencias ciudadanas contribuye a mejorar la convivencia escolar",
                "label": "NO CONTR"
            }
        ]
    },

    "cot": {

        "positivos": [
            {
                "texto": "primero se identifico que no existian diferencias estadisticamente significativas entre los grupos luego se concluyo que la hipotesis principal debia ser rechazada",
                "label": "CONTR"
            },
            {
                "texto": "aunque se esperaba una mejora en los indicadores de aprendizaje los resultados mostraron un descenso en el rendimiento academico de los estudiantes",
                "label": "CONTR"
            },
            {
                "texto": "el razonamiento derivado del estudio permitio establecer que la aplicacion del modelo teorico no coincide con la evidencia empirica observada",
                "label": "CONTR"
            },
            {
                "texto": "los investigadores determinaron que el incremento de recursos tecnologicos no produjo una mejora en las habilidades cognitivas evaluadas",
                "label": "CONTR"
            },
            {
                "texto": "despues del analisis comparativo se concluyo que las medidas implementadas no redujeron los niveles de riesgo operacional",
                "label": "CONTR"
            }
        ],

        "negativos": [
            {
                "texto": "primero se analizaron los datos obtenidos posteriormente se verifico que los resultados apoyaban la hipotesis formulada inicialmente",
                "label": "NO CONTR"
            },
            {
                "texto": "el estudio identifico mejoras en los procesos educativos y finalmente concluyo que las estrategias aplicadas fueron efectivas",
                "label": "NO CONTR"
            },
            {
                "texto": "la evidencia recopilada demuestra que la participacion comunitaria fortalece los procesos de gestion educativa en contextos escolares",
                "label": "NO CONTR"
            },
            {
                "texto": "los resultados permitieron confirmar que la implementacion de modelos agroecologicos favorece la biodiversidad en los sistemas productivos",
                "label": "NO CONTR"
            },
            {
                "texto": "el analisis realizado respalda la idea de que las tecnologias de informacion contribuyen al desarrollo de nuevas practicas pedagogicas",
                "label": "NO CONTR"
            }
        ]
    }
}

# =========================================================
# 14. API DEEPSEEK
# =========================================================

API_KEY = "sk-427c01e03b854dbbae3b791e765f9a65"

API_URL = "https://api.deepseek.com/chat/completions"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# =========================================================
# 15. FUNCION AUXILIAR
# =========================================================

def construir_ejemplos(strategy):

    ejemplos = ""

    positivos = PROMPT_EXAMPLES[strategy]["positivos"]
    negativos = PROMPT_EXAMPLES[strategy]["negativos"]

    for ej in positivos + negativos:

        ejemplos += (
            f"Texto: {ej['texto']}\n"
            f"Respuesta: {ej['label']}\n\n"
        )

    return ejemplos

# =========================================================
# 16. FUNCION DEEPSEEK
# =========================================================

def deepseek_predict(
    texto,
    pos_rel,
    strategy="zero-shot"
):

    ejemplos = construir_ejemplos(strategy)

    if strategy == "cot":

        prompt = f"""
Eres un clasificador experto de textos cientificos.

Ejemplos:

{ejemplos}

Analiza paso a paso:

1. Busca contradiccion.
2. Busca rechazo de hipotesis.
3. Decide la clase final.

Texto: {texto}

Posicion relativa: {pos_rel}

Responde SOLO:
CONTR
o
NO CONTR
"""

    elif strategy == "few-shot":

        prompt = f"""
Eres un clasificador cientifico.

Aprende de los ejemplos y clasifica.

Ejemplos:

{ejemplos}

Texto: {texto}

Posicion relativa: {pos_rel}

Responde SOLO:
CONTR
o
NO CONTR
"""

    else:

        prompt = f"""
Eres un clasificador de textos cientificos.

Clasifica el siguiente texto.

Ejemplos:

{ejemplos}

Texto: {texto}

Posicion relativa: {pos_rel}

Responde SOLO:
CONTR
o
NO CONTR
"""

    payload = {

        "model": "deepseek-chat",

        "messages": [

            {
                "role": "system",
                "content": "Eres un clasificador cientifico."
            },

            {
                "role": "user",
                "content": prompt
            }

        ],

        "temperature": 0.0
    }

    try:

        response = requests.post(
            API_URL,
            headers=headers,
            json=payload,
            timeout=60
        )

        if response.status_code != 200:

            print("\nERROR API")
            print(response.text)

            return 0

        result = response.json()

        salida = (
            result["choices"][0]
            ["message"]["content"]
            .strip()
            .upper()
        )

        if "NO CONTR" in salida:
            return 0

        elif "CONTR" in salida:
            return 1

        else:
            return 0

    except Exception as e:

        print(e)

        return 0

# =========================================================
# 17. EVALUACION DEEPSEEK
# =========================================================

def evaluar_deepseek(
    data_eval,
    estrategias=[
        "zero-shot",
        "few-shot",
        "cot"
    ]
):

    y_true = data_eval["label"].tolist()

    resultados = {}

    for strat in estrategias:

        print("\n================================================")
        print(f"Evaluando DeepSeek: {strat}")
        print("================================================")

        preds = []

        for i, (_, row) in enumerate(
            data_eval.iterrows()
        ):

            print(
                f"Procesando {i+1}/{len(data_eval)}"
            )

            pred = deepseek_predict(
                row["text"],
                row["pos_rel"],
                strategy=strat
            )

            preds.append(pred)

            time.sleep(1)

        resultados[strat] = preds

        evaluar_modelo(
            f"DeepSeek-{strat}",
            y_true,
            preds
        )

    return resultados

# =========================================================
# 18. EJECUTAR DEEPSEEK
# =========================================================

resultados_deepseek = evaluar_deepseek(
    eval_data,
    estrategias=[
        "zero-shot",
        "few-shot",
        "cot"
    ]
)

# =========================================================
# 19. COMPARACION CONCEPTUAL
# =========================================================

print("\n================================================")
print("COMPARACION CONCEPTUAL")
print("================================================")

print("""
Transfer Learning:
Usar embeddings preentrenados como BERT
y entrenar un clasificador encima.
""")

print("""
Fine-Tuning:
Ajustar todos los pesos del modelo.
Mayor costo computacional.
""")

print("""
DeepSeek Prompting:
Clasificacion directa mediante prompts.
No requiere entrenamiento adicional.
""")

print("""
LightGBM:
Modelo de boosting basado en arboles.
Excelente rendimiento en datasets tabulares.
""")

print("""
Red Neuronal Basica:
Modelo neuronal minimo sin capas ocultas.
Muy similar a una regresion logistica.
Sirve como baseline neuronal clasico.
""")


Distribucion de clases
label
0    2188
1    2172
Name: count, dtype: int64

Distribucion Dataset_type
Dataset_type
TRAIN    2775
TEST     1184
EVAL      401
Name: count, dtype: int64

TRAIN
2775

TEST
1184

EVAL
401

Entrenando LightGBM...
[LightGBM] [Info] Number of positive: 1385, number of negative: 1390
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.225259 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 123156
[LightGBM] [Info] Number of data points in the train set: 2775, number of used features: 4136
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499099 -> initscore=-0.003604
[LightGBM] [Info] Start training from score -0.003604

########################################
EVALUACION TEST
########################################

LightGBM TEST
Accuracy : 0.8767
Precision: 0.8704
Recall   : 0.8823
F1 Score : 0.8763
ROC-AUC  : 0.9

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:1238: UserWarning: Con


########################################
RED NEURONAL BASICA TEST
########################################

Red Neuronal Basica TEST
Accuracy : 0.7492
Precision: 0.7312
Recall   : 0.7799
F1 Score : 0.7547
ROC-AUC  : 0.8266

Reporte completo:

              precision    recall  f1-score   support

           0       0.77      0.72      0.74       598
           1       0.73      0.78      0.75       586

    accuracy                           0.75      1184
   macro avg       0.75      0.75      0.75      1184
weighted avg       0.75      0.75      0.75      1184


########################################
RED NEURONAL BASICA EVAL
########################################

Red Neuronal Basica EVAL
Accuracy : 0.7057
Precision: 0.7005
Recall   : 0.7214
F1 Score : 0.7108
ROC-AUC  : 0.7622

Reporte completo:

              precision    recall  f1-score   support

           0       0.71      0.69      0.70       200
           1       0.70      0.72      0.71       201

    accuracy         

In [7]:
# -*- coding: utf-8 -*-

"""
SISTEMA HIBRIDO:
LIGHTGBM + DEEPSEEK API

Incluye:
- LightGBM
- Exportacion de modelo:
    * joblib
    * pickle
    * txt
    * json
    * bin
- DeepSeek prompting:
    * Zero-shot
    * Few-shot
    * Chain of Thought (CoT)
- Ensemble hibrido:
    * 80% LightGBM
    * 20% DeepSeek

CORRECCIONES:
- NO usa train_test_split
- Usa Dataset_type:
    * TRAIN
    * TEST
    * EVAL
- Entrena SOLO con TRAIN
- Evalua TEST y EVAL
- DeepSeek usa EVAL
- Mantiene 30 ejemplos prompting
"""

# =========================================================
# 0. INSTALAR LIBRERIAS (SOLO COLAB)
# =========================================================

# !pip install lightgbm openpyxl -q

# =========================================================
# 1. LIBRERIAS
# =========================================================

import pandas as pd
import numpy as np
import requests
import scipy.sparse as sp
import time
import json
import pickle
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from sklearn.feature_extraction.text import TfidfVectorizer

import lightgbm as lgb

# =========================================================
# 2. CARGAR DATASET
# =========================================================

data = pd.read_excel(
    "DATASET_TAREA_2_CONSOLIDADO_FINAL_ORIGINAL.xlsx"
)

# =========================================================
# 3. LIMPIEZA BASICA
# =========================================================

data = data.dropna(
    subset=[
        "text",
        "label",
        "pos_rel",
        "Dataset_type"
    ]
)

data["text"] = data["text"].astype(str)

data["Dataset_type"] = (
    data["Dataset_type"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# =========================================================
# 4. LABEL BINARIO
# =========================================================

data["label"] = np.where(
    data["label"] == "CONTR",
    1,
    0
)

data["label"] = data["label"].astype(int)

print("\n===================================")
print("Distribucion de clases")
print("===================================")

print(
    data["label"].value_counts()
)

print("\n===================================")
print("Distribucion Dataset_type")
print("===================================")

print(
    data["Dataset_type"].value_counts()
)

# =========================================================
# 5. DIVISIONES DATASET
# =========================================================

train_data = data[
    data["Dataset_type"] == "TRAIN"
]

test_data = data[
    data["Dataset_type"] == "TEST"
]

eval_data = data[
    data["Dataset_type"] == "EVAL"
]

print("\n===================================")
print("TRAIN")
print("===================================")
print(len(train_data))

print("\n===================================")
print("TEST")
print("===================================")
print(len(test_data))

print("\n===================================")
print("EVAL")
print("===================================")
print(len(eval_data))

# =========================================================
# 6. TF-IDF
# =========================================================

vectorizer = TfidfVectorizer(
    max_features=5000
)

X_train_text = vectorizer.fit_transform(
    train_data["text"]
)

X_test_text = vectorizer.transform(
    test_data["text"]
)

X_eval_text = vectorizer.transform(
    eval_data["text"]
)

# =========================================================
# 7. FEATURE POS_REL
# =========================================================

X_train_pos = (
    train_data["pos_rel"]
    .values
    .reshape(-1, 1)
)

X_test_pos = (
    test_data["pos_rel"]
    .values
    .reshape(-1, 1)
)

X_eval_pos = (
    eval_data["pos_rel"]
    .values
    .reshape(-1, 1)
)

# =========================================================
# 8. MATRICES FINALES
# =========================================================

X_train = sp.hstack([
    X_train_text,
    X_train_pos
])

X_test = sp.hstack([
    X_test_text,
    X_test_pos
])

X_eval = sp.hstack([
    X_eval_text,
    X_eval_pos
])

y_train = train_data["label"]

y_test = test_data["label"]

y_eval = eval_data["label"]

# =========================================================
# 9. FUNCION METRICAS
# =========================================================

def evaluar_modelo(
    nombre,
    y_true,
    y_pred,
    y_proba=None
):

    print("\n================================================")
    print(nombre)
    print("================================================")

    print(
        "Accuracy :",
        round(
            accuracy_score(y_true, y_pred),
            4
        )
    )

    print(
        "Precision:",
        round(
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            4
        )
    )

    print(
        "Recall   :",
        round(
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            4
        )
    )

    print(
        "F1 Score :",
        round(
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            4
        )
    )

    if y_proba is not None:

        try:

            roc = roc_auc_score(
                y_true,
                y_proba
            )

            print(
                "ROC-AUC  :",
                round(roc, 4)
            )

        except:

            print("ROC-AUC no disponible")

    print("\nReporte completo:\n")

    print(
        classification_report(
            y_true,
            y_pred,
            zero_division=0
        )
    )

# =========================================================
# 10. LIGHTGBM
# =========================================================

print("\nEntrenando LightGBM...")

lgb_model = lgb.LGBMClassifier(

    random_state=42,

    n_estimators=300,

    learning_rate=0.05
)

lgb_model.fit(
    X_train,
    y_train
)

# =========================================================
# 11. EVALUACION LIGHTGBM TEST
# =========================================================

print("\n===================================")
print("LIGHTGBM TEST")
print("===================================")

y_pred_test = lgb_model.predict(
    X_test
)

y_proba_test = lgb_model.predict_proba(
    X_test
)[:, 1]

evaluar_modelo(
    "LightGBM TEST",
    y_test,
    y_pred_test,
    y_proba_test
)

# =========================================================
# 12. EVALUACION LIGHTGBM EVAL
# =========================================================

print("\n===================================")
print("LIGHTGBM EVAL")
print("===================================")

y_pred_eval = lgb_model.predict(
    X_eval
)

y_proba_eval = lgb_model.predict_proba(
    X_eval
)[:, 1]

evaluar_modelo(
    "LightGBM EVAL",
    y_eval,
    y_pred_eval,
    y_proba_eval
)

# =========================================================
# 13. EXPORTAR MODELO
# =========================================================

print("\nExportando modelos...")

joblib.dump(
    lgb_model,
    "lightgbm_model.joblib"
)

with open(
    "lightgbm_model.pkl",
    "wb"
) as f:

    pickle.dump(
        lgb_model,
        f
    )

lgb_model.booster_.save_model(
    "lightgbm_model.txt"
)

lgb_model.booster_.save_model(
    "lightgbm_model.bin"
)

modelo_json = lgb_model.booster_.dump_model()

with open(
    "lightgbm_model.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        modelo_json,
        f,
        indent=4
    )

joblib.dump(
    vectorizer,
    "tfidf_vectorizer.joblib"
)

print("\nModelos exportados correctamente.")

# =========================================================
# 14. 30 EJEMPLOS PROMPTING
# =========================================================

PROMPT_EXAMPLES = {

    "zero-shot": {

        "positivos": [
            {
                "texto": "los resultados obtenidos contradicen la hipotesis inicial debido a que no se encontro relacion significativa entre las variables de estudio en ninguno de los grupos evaluados",
                "label": "CONTR"
            },
            {
                "texto": "a diferencia de investigaciones previas el modelo aplicado no mejoro el rendimiento academico y presento resultados inferiores en comparacion con el metodo tradicional",
                "label": "CONTR"
            },
            {
                "texto": "los hallazgos muestran que la implementacion de estrategias agroecologicas no produjo el aumento esperado en la productividad de los cultivos evaluados",
                "label": "CONTR"
            },
            {
                "texto": "el analisis estadistico evidencio que la intervencion educativa aplicada no genero cambios significativos en las competencias ciudadanas de los estudiantes",
                "label": "CONTR"
            },
            {
                "texto": "los datos recopilados durante la investigacion refutan la teoria planteada inicialmente acerca de la relacion entre mineria y desarrollo institucional",
                "label": "CONTR"
            }
        ],

        "negativos": [
            {
                "texto": "los resultados permitieron confirmar que el uso de tecnologias educativas favorece el aprendizaje colaborativo en ambientes virtuales",
                "label": "NO CONTR"
            },
            {
                "texto": "el estudio concluye que la implementacion de sistemas de control interno mejora la eficiencia administrativa y reduce riesgos financieros",
                "label": "NO CONTR"
            },
            {
                "texto": "la investigacion demuestra que la diversidad vegetal en sistemas agroecologicos favorece la presencia de organismos beneficos",
                "label": "NO CONTR"
            },
            {
                "texto": "los autores encontraron evidencia suficiente para respaldar la relacion positiva entre participacion familiar y desarrollo cognitivo infantil",
                "label": "NO CONTR"
            },
            {
                "texto": "el analisis de los datos confirma que las estrategias pedagogicas aplicadas fortalecieron el proceso de aprendizaje de los estudiantes",
                "label": "NO CONTR"
            }
        ]
    },

    "few-shot": {

        "positivos": [
            {
                "texto": "contrario a lo reportado en investigaciones anteriores la aplicacion del modelo matematico no permitio mejorar la estabilidad del vehiculo",
                "label": "CONTR"
            },
            {
                "texto": "los resultados experimentales no respaldan la teoria de que el incremento de concesiones mineras fortalece la institucionalidad regional",
                "label": "CONTR"
            },
            {
                "texto": "el estudio evidencio que las practicas de control propuestas no redujeron los costos operativos como se esperaba inicialmente",
                "label": "CONTR"
            },
            {
                "texto": "a pesar de las expectativas iniciales el uso de plataformas virtuales no incremento la participacion activa de los estudiantes en los foros",
                "label": "CONTR"
            },
            {
                "texto": "la investigacion concluye que la intervencion aplicada no tuvo impacto significativo sobre la reduccion de accidentes de motociclistas",
                "label": "CONTR"
            }
        ],

        "negativos": [
            {
                "texto": "los hallazgos respaldan la importancia de incorporar estrategias de responsabilidad social empresarial en las politicas gubernamentales",
                "label": "NO CONTR"
            },
            {
                "texto": "los resultados obtenidos muestran una mejora considerable en la interaccion social de los estudiantes despues de aplicar la estrategia educativa",
                "label": "NO CONTR"
            },
            {
                "texto": "el estudio confirma que la utilizacion de sensores inteligentes mejora la estabilidad y seguridad en los sistemas de control vehicular",
                "label": "NO CONTR"
            },
            {
                "texto": "la investigacion evidencia que los sistemas agroecologicos presentan una mayor diversidad biologica que los sistemas convencionales",
                "label": "NO CONTR"
            },
            {
                "texto": "los autores concluyen que el fortalecimiento de las competencias ciudadanas contribuye a mejorar la convivencia escolar",
                "label": "NO CONTR"
            }
        ]
    },

    "cot": {

        "positivos": [
            {
                "texto": "primero se identifico que no existian diferencias estadisticamente significativas entre los grupos luego se concluyo que la hipotesis principal debia ser rechazada",
                "label": "CONTR"
            },
            {
                "texto": "aunque se esperaba una mejora en los indicadores de aprendizaje los resultados mostraron un descenso en el rendimiento academico de los estudiantes",
                "label": "CONTR"
            },
            {
                "texto": "el razonamiento derivado del estudio permitio establecer que la aplicacion del modelo teorico no coincide con la evidencia empirica observada",
                "label": "CONTR"
            },
            {
                "texto": "los investigadores determinaron que el incremento de recursos tecnologicos no produjo una mejora en las habilidades cognitivas evaluadas",
                "label": "CONTR"
            },
            {
                "texto": "despues del analisis comparativo se concluyo que las medidas implementadas no redujeron los niveles de riesgo operacional",
                "label": "CONTR"
            }
        ],

        "negativos": [
            {
                "texto": "primero se analizaron los datos obtenidos posteriormente se verifico que los resultados apoyaban la hipotesis formulada inicialmente",
                "label": "NO CONTR"
            },
            {
                "texto": "el estudio identifico mejoras en los procesos educativos y finalmente concluyo que las estrategias aplicadas fueron efectivas",
                "label": "NO CONTR"
            },
            {
                "texto": "la evidencia recopilada demuestra que la participacion comunitaria fortalece los procesos de gestion educativa en contextos escolares",
                "label": "NO CONTR"
            },
            {
                "texto": "los resultados permitieron confirmar que la implementacion de modelos agroecologicos favorece la biodiversidad en los sistemas productivos",
                "label": "NO CONTR"
            },
            {
                "texto": "el analisis realizado respalda la idea de que las tecnologias de informacion contribuyen al desarrollo de nuevas practicas pedagogicas",
                "label": "NO CONTR"
            }
        ]
    }
}

# =========================================================
# 15. API DEEPSEEK
# =========================================================

API_KEY = "TU_API_KEY"

API_URL = "https://api.deepseek.com/chat/completions"

headers = {

    "Authorization": f"Bearer {API_KEY}",

    "Content-Type": "application/json"
}

# =========================================================
# 16. FUNCION AUXILIAR
# =========================================================

def construir_ejemplos(strategy):

    ejemplos = ""

    positivos = PROMPT_EXAMPLES[strategy]["positivos"]
    negativos = PROMPT_EXAMPLES[strategy]["negativos"]

    for ej in positivos + negativos:

        ejemplos += (
            f"Texto: {ej['texto']}\n"
            f"Respuesta: {ej['label']}\n\n"
        )

    return ejemplos

# =========================================================
# 17. DEEPSEEK PREDICT
# =========================================================

def deepseek_predict_proba(
    texto,
    pos_rel,
    strategy="cot"
):

    ejemplos = construir_ejemplos(strategy)

    if strategy == "cot":

        prompt = f"""
Eres un clasificador experto de textos cientificos.

Ejemplos:

{ejemplos}

Analiza paso a paso:

1. Busca contradiccion.
2. Busca rechazo de hipotesis.
3. Busca resultados negativos.
4. Decide la clase final.

Texto: {texto}

Posicion relativa: {pos_rel}

Responde SOLO:
CONTR
o
NO CONTR
"""

    elif strategy == "few-shot":

        prompt = f"""
Eres un clasificador cientifico.

Aprende de los ejemplos y clasifica.

Ejemplos:

{ejemplos}

Texto: {texto}

Posicion relativa: {pos_rel}

Responde SOLO:
CONTR
o
NO CONTR
"""

    else:

        prompt = f"""
Eres un clasificador cientifico.

Clasifica el siguiente texto.

Ejemplos:

{ejemplos}

Texto: {texto}

Posicion relativa: {pos_rel}

Responde SOLO:
CONTR
o
NO CONTR
"""

    payload = {

        "model": "deepseek-chat",

        "messages": [

            {
                "role": "system",
                "content": "Eres un clasificador cientifico."
            },

            {
                "role": "user",
                "content": prompt
            }

        ],

        "temperature": 0.0
    }

    try:

        response = requests.post(
            API_URL,
            headers=headers,
            json=payload,
            timeout=60
        )

        result = response.json()

        salida = (
            result["choices"][0]
            ["message"]["content"]
            .strip()
            .upper()
        )

        if "NO CONTR" in salida:
            return 0.0

        elif "CONTR" in salida:
            return 1.0

        else:
            return 0.0

    except Exception as e:

        print(e)

        return 0.0

# =========================================================
# 18. ENSAMBLE HIBRIDO
# =========================================================

def hybrid_predict(

    texto,

    pos_rel,

    peso_lgbm=0.8,

    peso_deepseek=0.2,

    strategy="cot"
):

    X_text = vectorizer.transform(
        [texto]
    )

    X_pos = np.array(
        [[pos_rel]]
    )

    X_input = sp.hstack([
        X_text,
        X_pos
    ])

    prob_lgbm = lgb_model.predict_proba(
        X_input
    )[0][1]

    prob_deepseek = deepseek_predict_proba(
        texto,
        pos_rel,
        strategy=strategy
    )

    prob_final = (

        peso_lgbm * prob_lgbm

        +

        peso_deepseek * prob_deepseek
    )

    pred_final = int(
        prob_final >= 0.5
    )

    return {

        "prob_lgbm": round(prob_lgbm, 4),

        "prob_deepseek": round(prob_deepseek, 4),

        "prob_final": round(prob_final, 4),

        "pred_final": pred_final
    }

# =========================================================
# 19. EVALUACION HIBRIDA EVAL
# =========================================================

print("\n===================================")
print("EVALUACION HIBRIDA EVAL")
print("===================================")

y_true = eval_data["label"].tolist()

preds_hybrid = []

for i, (_, row) in enumerate(
    eval_data.iterrows()
):

    print(
        f"Procesando {i+1}/{len(eval_data)}"
    )

    resultado = hybrid_predict(

        texto=row["text"],

        pos_rel=row["pos_rel"],

        peso_lgbm=0.8,

        peso_deepseek=0.2,

        strategy="cot"
    )

    preds_hybrid.append(
        resultado["pred_final"]
    )

    time.sleep(1)

# =========================================================
# 20. METRICAS HIBRIDAS
# =========================================================

evaluar_modelo(
    "Hybrid LightGBM + DeepSeek",
    y_true,
    preds_hybrid
)

# =========================================================
# 21. EJEMPLO INDIVIDUAL
# =========================================================

resultado = hybrid_predict(

    texto="""
    los resultados contradicen la teoria principal
    y no se encontraron mejoras significativas
    """,

    pos_rel=0.82,

    strategy="cot"
)

print("\n===================================")
print("EJEMPLO PREDICCION")
print("===================================")

print(resultado)

# =========================================================
# 22. COMPARACION CONCEPTUAL
# =========================================================

print("\n================================================")
print("COMPARACION CONCEPTUAL")
print("================================================")

print("""
LightGBM:
Modelo estadistico supervisado basado en boosting.
Excelente para datasets tabulares y TF-IDF.
""")

print("""
DeepSeek Prompting:
Modelo semantico basado en lenguaje natural.
Capaz de detectar contradicciones implicitas.
""")

print("""
Sistema Hibrido:
Combina:
- precision estadistica
- comprension semantica

LightGBM aporta robustez numerica.
DeepSeek aporta comprension contextual.
""")

print("""
Este enfoque corresponde a:

LLM-Augmented Machine Learning
o
Hybrid Neuro-Symbolic AI
""")


Distribucion de clases
label
0    2188
1    2172
Name: count, dtype: int64

Distribucion Dataset_type
Dataset_type
TRAIN    2775
TEST     1184
EVAL      401
Name: count, dtype: int64

TRAIN
2775

TEST
1184

EVAL
401

Entrenando LightGBM...
[LightGBM] [Info] Number of positive: 1385, number of negative: 1390
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.135024 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 123156
[LightGBM] [Info] Number of data points in the train set: 2775, number of used features: 4136
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499099 -> initscore=-0.003604
[LightGBM] [Info] Start training from score -0.003604

LIGHTGBM TEST

LightGBM TEST
Accuracy : 0.886
Precision: 0.8803
Recall   : 0.8908
F1 Score : 0.8855
ROC-AUC  : 0.9569

Reporte completo:

              precision    recall  f1-score   support

           0       0.89      0.88      0.89       598
           1    

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:1238: UserWarning: Con


Modelos exportados correctamente.

EVALUACION HIBRIDA EVAL
Procesando 1/401
'choices'


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Procesando 2/401
'choices'
Procesando 3/401
'choices'
Procesando 4/401
'choices'
Procesando 5/401
'choices'
Procesando 6/401
'choices'
Procesando 7/401
'choices'
Procesando 8/401
'choices'
Procesando 9/401
'choices'
Procesando 10/401
'choices'
Procesando 11/401
'choices'
Procesando 12/401
'choices'
Procesando 13/401
'choices'
Procesando 14/401
'choices'
Procesando 15/401
'choices'
Procesando 16/401
'choices'
Procesando 17/401
'choices'
Procesando 18/401
'choices'
Procesando 19/401
'choices'
Procesando 20/401
'choices'
Procesando 21/401
'choices'
Procesando 22/401
'choices'
Procesando 23/401
'choices'
Procesando 24/401
'choices'
Procesando 25/401
'choices'
Procesando 26/401
'choices'
Procesando 27/401
'choices'
Procesando 28/401
'choices'
Procesando 29/401
'choices'
Procesando 30/401
'choices'
Procesando 31/401
'choices'
Procesando 32/401
'choices'
Procesando 33/401
'choices'
Procesando 34/401
'choices'
Procesando 35/401
'choices'
Procesando 36/401
'choices'
Procesando 37/401
'choices'


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")
